# PCT Training - Occlusion

Trains Menghao PCT model from the Point-Transformers implementation: https://github.com/qq456cvb/Point-Transformers

Changes: Added an occlusion function that randomly drops between 0.1 to 0.9 fraction of points from the cloud in a plane-based manner

Data: fullmodelnet40 version

## Env prep

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Paths, folders
import os

REPO_PATH = '/content/pointcloud-bench'
DRIVE_PATH = '/content/drive/MyDrive/pointcloud-bench'
results_dir = os.path.join(DRIVE_PATH, 'results')
os.makedirs(results_dir, exist_ok=True)

In [ ]:
# Get repo
!git clone --recurse-submodules --branch pct-plots https://github.com/DavidClaszen/pointcloud-bench {REPO_PATH}

# Submodule handling
%cd {REPO_PATH}
!git submodule update --init --recursive
%cd repos/Point-Transformers
!git fetch origin pct-occlusion
!git checkout pct-occlusion
!git pull origin pct-occlusion
%cd {REPO_PATH}

%pip install -r envs/pct/requirements.txt

Cloning into '/content/pointcloud-bench'...
remote: Enumerating objects: 467, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 467 (delta 85), reused 111 (delta 43), pack-reused 292 (from 1)
Receiving objects: 100% (467/467), 40.04 MiB | 41.29 MiB/s, done.
Resolving deltas: 100% (201/201), done.
Submodule 'repos/PAPNet' (https://github.com/DavidClaszen/PAPNet.git) registered for path 'repos/PAPNet'
Submodule 'repos/Point-Transformers' (https://github.com/DavidClaszen/Point-Transformers.git) registered for path 'repos/Point-Transformers'
Cloning into '/content/pointcloud-bench/repos/PAPNet'...
remote: Enumerating objects: 133, done.        
remote: Counting objects: 100% (133/133), done.        
remote: Compressing objects: 100% (105/105), done.        
remote: Total 133 (delta 55), reused 85 (delta 27), pack-reused 0 (from 0)        
Receiving objects: 100% (133/133), 9.66 MiB | 14.50 MiB/s, done.
Resolving deltas: 1

In [ ]:
# Check for CUDA/GPU
import torch, sys
print(sys.version)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.is_available())

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.9.0+cu126 CUDA: 12.6 GPU: True


In [ ]:
# Copy and unzip only the fullmodelnet40 set
# Evaluation will be done in other notebook
!rsync -avP {DRIVE_PATH}/datasets/fullmodelnet40.tar.gz {REPO_PATH}/datasets
%cd {REPO_PATH}
!tar -xvzf datasets/fullmodelnet40.tar.gz -C datasets

sending incremental file list
fullmodelnet40.tar.gz
  1,833,210,387 100%   83.89MB/s    0:00:20 (xfr#1, to-chk=0/1)

sent 1,833,658,058 bytes  received 35 bytes  85,286,422.93 bytes/sec
total size is 1,833,210,387  speedup is 1.00
/content/pointcloud-bench
fullmodelnet40/
fullmodelnet40/test_filenames.txt
fullmodelnet40/test_gt_rot.npy
fullmodelnet40/test_gt_tra.npy
fullmodelnet40/test_labels.npy
fullmodelnet40/test_points.npy
fullmodelnet40/train_filenames.txt
fullmodelnet40/train_gt_rot.npy
fullmodelnet40/train_gt_tra.npy
fullmodelnet40/train_labels.npy
fullmodelnet40/train_points.npy


# Model Training

Since we're only using PAPNet style data here, always set `use_papnet_loader` to `True`.

New arguments:
- occlusion: bool
- occlusion_min: float, default 0.1
- occlusion_max: float, default 0.9

But showing it full clouds somtimes is probably a good idea, so we'll set min to 0.01


In [ ]:
%cd /content/pointcloud-bench/repos/Point-Transformers
!python train_cls.py --help

/content/pointcloud-bench/repos/Point-Transformers
train_cls is powered by Hydra.

== Configuration groups ==
Compose your configuration from those groups (group=option)

model: Hengshuang, Menghao, Nico


== Config ==
Override anything in the config (foo.bar=value)

model:
  name: Menghao
batch_size: 16
epoch: 200
learning_rate: 0.001
gpu: 0
num_point: 1024
optimizer: Adam
weight_decay: 0.0001
normal: true
use_papnet_loader: false
workers: 2
step_size: 50
data_path: ../../datasets/modelnet40_normal_resampled/
checkpoint_path: best_model.pth
partiality: ''
occlusion: ''
occlusion_min: 0.1
occlusion_max: 0.9


Powered by Hydra (https://hydra.cc)
Use --hydra-help to view Hydra specific help




In [ ]:
# Train Menghao
!python train_cls.py model=Menghao use_papnet_loader=True batch_size=512 learning_rate=0.0005 epoch=50 workers=4 step_size=20 data_path=../../datasets/fullmodelnet40/ occlusion=True occlusion_min=0.01 occlusion_max=0.9

model:
  name: Menghao
batch_size: 512
epoch: 50
learning_rate: 0.0005
gpu: 0
num_point: 1024
optimizer: Adam
weight_decay: 0.0001
normal: true
use_papnet_loader: true
workers: 4
step_size: 20
data_path: ../../datasets/fullmodelnet40/
checkpoint_path: best_model.pth
partiality: ''
occlusion: true
occlusion_min: 0.01
occlusion_max: 0.9

[2025-12-02 09:40:56,825][__main__][INFO] - Load dataset ...
The size of train data is 98430
The size of test data is 2468
[2025-12-02 09:40:57,726][__main__][INFO] - No existing model, starting training from scratch...
[2025-12-02 09:40:58,718][__main__][INFO] - Start training...
[2025-12-02 09:40:58,718][__main__][INFO] - Epoch 1 (1/50):
100% 193/193 [02:28<00:00,  1.30it/s]
[2025-12-02 09:43:27,639][__main__][INFO] - Train Instance Accuracy: 0.255266
100% 5/5 [00:05<00:00,  1.14s/it]
[2025-12-02 09:43:33,406][__main__][INFO] - Test Instance Accuracy: 0.374249, Class Accuracy: 0.276005
[2025-12-02 09:43:33,406][__main__][INFO] - Best Instance Accuracy:

In [ ]:
# Zip Point-Transformers logs, included last best model
!zip -r results.zip ./log/cls/Menghao/

  adding: log/cls/Menghao/ (stored 0%)
  adding: log/cls/Menghao/model.py (deflated 76%)
  adding: log/cls/Menghao/train_cls.log (deflated 88%)
  adding: log/cls/Menghao/.hydra/ (stored 0%)
  adding: log/cls/Menghao/.hydra/config.yaml (deflated 34%)
  adding: log/cls/Menghao/.hydra/overrides.yaml (deflated 29%)
  adding: log/cls/Menghao/.hydra/hydra.yaml (deflated 66%)
